In [ ]:
import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score,
    make_scorer
)
from sklearn.model_selection import KFold

In [ ]:
from xgboost import XGBRegressor
from skopt import BayesSearchCV
from skopt.space import Real, Integer

In [ ]:
RANDOM_STATE = 42
USE_GPU = True  # set based on session (Kaggle GPU: True, local CPU: False)
N_COMPONENTS = 20  # fixed, from prior PCA-dimensionality ablation

In [ ]:
# ---- 1. Load data ----
DATA_DIR = "/kaggle/input/kickstarter-joint-embeddings"  # <-- update as needed

In [ ]:
TRAIN_FILE = f"{DATA_DIR}/ML_train.csv"
TEST_FILE = f"{DATA_DIR}/ML_test.csv"

In [ ]:
POOLING_FILES = {
    "CLS":  f"{DATA_DIR}/bert_cls_embeddings.csv",
    "Mean": f"{DATA_DIR}/bert_mean_embeddings.csv",   # == existing bert_embeddings.csv
    "Max":  f"{DATA_DIR}/bert_max_embeddings.csv",
}

In [ ]:
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

In [ ]:
TARGET_RAW = "target_usd"
TARGET_LOG = "log_target"
DROP_FROM_X = ["id", "target_usd", "log_target", "goal_usd"]

In [ ]:
train_ids = set(train_df["id"])

In [ ]:
# ---- 2. Evaluation helper ----
def evaluate_model(y_true_log, pred_log, actual_usd):
    pred_usd = np.expm1(pred_log)
    pred_usd = np.clip(pred_usd, a_min=0, a_max=None)

    mae_log = mean_absolute_error(y_true_log, pred_log)
    mse_log = mean_squared_error(y_true_log, pred_log)
    rmse_log = np.sqrt(mse_log)
    r2_log = r2_score(y_true_log, pred_log)

    mae_usd = mean_absolute_error(actual_usd, pred_usd)
    mse_usd = mean_squared_error(actual_usd, pred_usd)
    rmse_usd = np.sqrt(mse_usd)
    r2_usd = r2_score(actual_usd, pred_usd)

    rmsle = np.sqrt(mean_squared_log_error(actual_usd, pred_usd))

    return {
        "MAE_log": mae_log, "MSE_log": mse_log, "RMSE_log": rmse_log, "R2_log": r2_log,
        "MAE_USD": mae_usd, "MSE_USD": mse_usd, "RMSE_USD": rmse_usd, "R2_USD": r2_usd,
        "RMSLE": rmsle
    }

In [ ]:
# ---- 3. BayesSearchCV setup (fixed across all pooling types) ----
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
rmse_scorer = make_scorer(rmse, greater_is_better=False)
CV = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
N_ITER_BAYES = 4

In [ ]:
xgb_bayes_space = {
    "n_estimators":     Integer(200, 500),
    "learning_rate":    Real(0.01, 0.1, prior="log-uniform"),
    "max_depth":        Integer(4, 9),
    "min_child_weight": Integer(1, 7),
    "subsample":        Real(0.7, 1.0),
    "colsample_bytree": Real(0.7, 1.0),
    "reg_alpha":        Real(1e-3, 0.5, prior="log-uniform"),
    "reg_lambda":       Real(0.5, 3.0),
}

In [ ]:
def make_search():
    base_estimator = XGBRegressor(
        objective="reg:squarederror", eval_metric="rmse",
        tree_method="hist", device="cuda" if USE_GPU else "cpu",
        random_state=RANDOM_STATE,
    )
    return BayesSearchCV(
        estimator=base_estimator, search_spaces=xgb_bayes_space, n_iter=N_ITER_BAYES,
        scoring=rmse_scorer, cv=CV, n_jobs=1,
        random_state=RANDOM_STATE, verbose=0, refit=True,
    )

In [ ]:
# ---- 4. Ablation loop over pooling types ----
ablation_results = []
best_params_by_pool = {}

In [ ]:
for pool_name, path in POOLING_FILES.items():
    print(f"\n{'='*70}")
    print(f"POOLING = {pool_name}")
    print(f"{'='*70}")

    pool_raw = pd.read_csv(path)
    feature_cols = [c for c in pool_raw.columns if c != "id"]
    train_mask = pool_raw["id"].isin(train_ids)

    pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
    pca.fit(pool_raw.loc[train_mask, feature_cols])
    reduced_all = pca.transform(pool_raw[feature_cols])

    reduced_cols = [f"bert_pca_{i}" for i in range(N_COMPONENTS)]
    reduced_df = pd.DataFrame(reduced_all, columns=reduced_cols)
    reduced_df.insert(0, "id", pool_raw["id"].values)

    explained = pca.explained_variance_ratio_.sum()
    print(f"Explained variance ({pool_name}): {explained:.3f}")

    merged_train = train_df.merge(reduced_df, on="id", how="left")
    merged_test = test_df.merge(reduced_df, on="id", how="left")

    assert merged_train.shape[0] == train_df.shape[0]
    assert merged_test.shape[0] == test_df.shape[0]
    assert merged_train[reduced_cols].isna().sum().sum() == 0
    assert merged_test[reduced_cols].isna().sum().sum() == 0

    X_train = merged_train.drop(columns=DROP_FROM_X, errors="ignore")
    X_test = merged_test.drop(columns=DROP_FROM_X, errors="ignore")
    y_train = merged_train[TARGET_LOG].copy()
    y_test = merged_test[TARGET_LOG].copy()
    actual_usd = merged_test[TARGET_RAW].to_numpy()

    assert list(X_train.columns) == list(X_test.columns)

    search = make_search()
    start_time = time.time()
    search.fit(X_train, y_train)
    tuning_time = time.time() - start_time

    best_model = search.best_estimator_
    pred_log = best_model.predict(X_test)

    metrics = evaluate_model(y_test, pred_log, actual_usd)
    metrics["Pooling"] = pool_name
    metrics["Explained_Variance"] = explained
    metrics["Best_CV_RMSE_log"] = -search.best_score_
    metrics["Tuning_Time_Seconds"] = tuning_time

    ablation_results.append(metrics)
    best_params_by_pool[pool_name] = dict(search.best_params_)

    print(f"RMSLE={metrics['RMSLE']:.4f} | R2_log={metrics['R2_log']:.4f} | "
          f"R2_USD={metrics['R2_USD']:.4f} | time={tuning_time:.1f}s")

In [ ]:
# ---- 5. Results table ----
results_df = pd.DataFrame(ablation_results)
results_df = results_df[[
    "Pooling", "Explained_Variance",
    "MAE_log", "MSE_log", "RMSE_log", "R2_log",
    "MAE_USD", "MSE_USD", "RMSE_USD", "R2_USD",
    "RMSLE", "Best_CV_RMSE_log", "Tuning_Time_Seconds"
]]
print("\n" + "="*70)
print("ABLATION RESULTS: BERT pooling strategy (CLS vs Mean vs Max)")
print("="*70)
print(results_df)

In [ ]:
results_df.to_csv("bert_pooling_ablation_results.csv", index=False)
with open("bert_pooling_ablation_best_params.json", "w") as f:
    json.dump(best_params_by_pool, f, indent=2, default=str)

In [ ]:
print("\nSaved: bert_pooling_ablation_results.csv")
print("Saved: bert_pooling_ablation_best_params.json")

In [ ]:
# ---- 6. Plot ----
import matplotlib.pyplot as plt

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

In [ ]:
axes[0].bar(results_df["Pooling"], results_df["RMSLE"])
axes[0].set_ylabel("RMSLE")
axes[0].set_title("RMSLE by Pooling Strategy")
axes[0].grid(True, alpha=0.3, axis="y")

In [ ]:
x = np.arange(len(results_df))
width = 0.35
axes[1].bar(x - width/2, results_df["R2_log"], width, label="R2_log")
axes[1].bar(x + width/2, results_df["R2_USD"], width, label="R2_USD")
axes[1].set_xticks(x)
axes[1].set_xticklabels(results_df["Pooling"])
axes[1].set_ylabel("R2")
axes[1].set_title("R2 by Pooling Strategy")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

In [ ]:
plt.tight_layout()
plt.savefig("bert_pooling_ablation_plot.png", dpi=150)
plt.show()
print("Saved: bert_pooling_ablation_plot.png")